# Speculative Decoding

**难度:** Hard

实现推测解码以加速 LLM 推理。

推测解码使用快速草稿模型提议 token，然后根据概率比率与目标模型验证，接受或重新采样。

**签名:** `speculative_decode(target_probs, draft_probs, draft_tokens) -> list[int]`

**参数:**
- `target_probs` — 目标模型概率 (K, V)
- `draft_probs` — 草稿模型概率 (K, V)
- `draft_tokens` — 提议的 token ID (K,)

**返回:** 接受的 token ID 列表（长度 1 到 K）

**约束:**
- 以概率 `min(1, p_target/p_draft)` 接受
- 拒绝时从归一化的 `max(0, p_target - p_draft)` 重新采样
- 在首次拒绝时停止（返回已接受的 + 重采样 token）

**提示:**
for i in range(K):
  accept_prob = min(1, p_target[i, token] / p_draft[i, token])
  若接受：追加 token，继续
  若拒绝：从归一化的 max(0, p_target[i] - p_draft[i]) 重采样，追加后返回

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [ ]:
# ✅ INTERVIEW

# ---- 手写推测解码 ----
# 面试要点：理解接受-拒绝采样的数学原理

def speculative_decode(target_probs, draft_probs, draft_tokens):
    K = len(draft_tokens)
    accepted = []

    for i in range(K):
        t = draft_tokens[i].item()

        # 面试考点：接受概率的推导
        # 目标：从 p_target 采样，但不想跑 K 次大模型
        # 方法：用 p_draft 提议，以 min(1, p_target/p_draft) 接受
        # 数学保证：接受的 token 的分布恰好是 p_target
        ratio = target_probs[i, t] / max(draft_probs[i, t].item(), 1e-10)

        if torch.rand(1).item() < min(1.0, ratio.item()):
            accepted.append(t)
        else:
            # 面试考点：拒绝后的修正分布
            # p_corrected(x) ∝ max(0, p_target(x) - p_draft(x))
            # 为什么？推导：
            # P(输出=x) = P(接受x) + P(拒绝)·P(重采样x)
            #           = p_draft(x)·min(1, p_target(x)/p_draft(x)) + Σ_reject·p_corrected(x)
            # 经过化简 = p_target(x) ✓
            adjusted = torch.clamp(target_probs[i] - draft_probs[i], min=0)
            s = adjusted.sum()
            if s > 0:
                adjusted = adjusted / s
            else:
                adjusted = torch.ones_like(adjusted) / adjusted.shape[0]
            accepted.append(torch.multinomial(adjusted, 1).item())
            return accepted  # 首次拒绝后停止

    return accepted

In [ ]:
# Demo
torch.manual_seed(0)
probs = torch.softmax(torch.randn(4, 10), dim=-1)
tokens = torch.tensor([2, 5, 1, 8])
print('Perfect draft:', speculative_decode(probs, probs, tokens))

In [ ]:
from torch_judge import check
check('speculative_decoding')

## 📝 核心思路总结

1. **推测解码的核心思想**：用小模型快速"猜"多个 token，大模型一次性验证，拒绝时修正
2. **接受概率 = min(1, p_target/p_draft)**：草稿模型概率越高越容易被拒绝（防止偏差）
3. **拒绝时的修正分布**：max(0, p_target - p_draft) 确保最终输出分布与纯目标模型一致